# Case Study I: 2024 Henada Airport Runway Incursion:

- On 2 January 2024, a runway collision occurred at Haneda Airport in Tokyo, Japan, involving an Airbus A350-900, operating as Japan Airlines Flight 516 (JAL516), and a De Havilland Canada Dash 8-Q300 operated by the Japan Coast Guard (JA722A). Japan Airlines Flight 516 was a scheduled domestic passenger flight from New Chitose Airport near Sapporo, Japan, to Haneda Airport in Tokyo. The Coast Guard plane was scheduled to deliver relief supplies a day after the 2024 Noto earthquake.

In [1]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)
import spacy

nlp_ner = spacy.load("./transformer/model-best")
ruler = nlp_ner.add_pipe(
    "entity_ruler",
    after="ner",  # or before="ner"
    config={"overwrite_ents": True}  
)
ruler.from_disk("entity_rulers.jsonl")

text = "Japan Air 179, Tokyo Tower, good evening, number 3, taxi to holding point C1."
doc = nlp_ner(text)

print("Entities found:")
for ent in doc.ents:
    print(f" - {ent.text} : {ent.label_}")


Entities found:
 - Japan Air 179 : CALLSIGN
 - taxi : ACSTATE
 - holding point C1 : DESTINATION


In [21]:
import re
import glob
import pandas as pd
import spacy
import torch
import numpy as np
from sentence_transformers import SentenceTransformer, util

###############################################################################
# 1. Runway Pattern Functions
###############################################################################
# This regex finds an optional "runway" followed by 1–2 digits and an optional L/R.
# It requires a whitespace, end-of-string, or common punctuation after the designator.
RUNWAY_REGEX = re.compile(r'(?i)(?:runway\s+)?(\d{1,2}[LR])(?=\s|$|[.,;:!])')

def extract_all_runway_designators(text: str) -> list:
    """
    Return all runway designators found in the text as a list.
    The designator must be one or two digits followed immediately by either L or R.
    
    For example:
      "17:43:12 ... runway 34R, continue approach" 
    will return: ["34R"]
    
    All designators are returned in uppercase.
    """
    matches = RUNWAY_REGEX.findall(text)
    return [m.upper() for m in matches if m.strip()]

def is_runway_pattern(text: str) -> bool:
    """Return True if at least one runway designator is found in the text."""
    return len(extract_all_runway_designators(text)) > 0

def format_runway_code(designator: str) -> str:
    """
    Format the runway designator by prepending 'RW'.
    For example, "8R" becomes "RW8R".
    """
    return f"RW{designator}"

def find_runway_entry_node(df: pd.DataFrame, runway_designator: str) -> str | None:
    """
    Given a runway designator (e.g. "8R"), look up the CSV row where that runway
    appears as an entry. In the CSV we assume that the runway entry is indicated by:
      - Either: refName1 equals the formatted runway code and type1 equals "Entry"
      - Or: refName2 equals the formatted runway code and type2 equals "Entry"
    Returns the node 'id' of the first matching row, or None if not found.
    """
    runway_code = format_runway_code(runway_designator)  # e.g. "RW8R"
    mask_ref1 = (df['refName1'] == runway_code) & (df['type1'].str.lower() == 'entry')
    mask_ref2 = (df['refName2'] == runway_code) & (df['type2'].str.lower() == 'entry')
    match_mask = mask_ref1 | mask_ref2
    matched_rows = df[match_mask]
    if not matched_rows.empty:
        return matched_rows.iloc[0]['id']
    return None

###############################################################################
# 2. Node Similarity Functions for Non-Runway Destinations
###############################################################################
def build_id_embeddings(df: pd.DataFrame, model_name='sentence-transformers/all-MiniLM-L6-v2'):
    """
    Build embeddings for the 'id' column of the CSV.
    Returns a tuple (model, embeddings).
    """
    model = SentenceTransformer(model_name)
    texts = df['id'].astype(str).tolist()
    embeddings = model.encode(texts, convert_to_tensor=True)
    return model, embeddings

def find_topk_similar_nodes(query: str, df: pd.DataFrame, model, embeddings, top_k=5):
    """
    Compute top-k similarity between the query and the embedded node IDs.
    Returns a list of dictionaries with the node id and similarity score.
    """
    query_emb = model.encode(query, convert_to_tensor=True)
    cos_scores = util.cos_sim(query_emb, embeddings)[0]
    top_results = torch.topk(cos_scores, k=top_k)
    results = []
    all_ids = df['id'].tolist()
    for i, idx_tensor in enumerate(top_results.indices):
        idx = idx_tensor.item()
        score = top_results.values[i].item()
        results.append({
            "id": all_ids[idx],
            "similarity_score": float(score)
        })
    return results

###############################################################################
# 3. Main Function: Build the Meta Table from the Transcript
###############################################################################
if __name__ == "__main__":
    # --- Load your pretrained spaCy model and add your entity ruler ---
    nlp_ner = spacy.load("./transformer/model-best")
    # Add the entity ruler (adjust the pipe position as needed)
    ruler = nlp_ner.add_pipe("entity_ruler", after="ner", config={"overwrite_ents": True})
    ruler.from_disk("entity_rulers.jsonl")
    
    # --- Load the CSV with airport nodes ---
    ICAO = 'HND'
    airport_nodes = f'./Airport Layouts/{ICAO}_Nodes_Def.csv'
    df = pd.read_csv(airport_nodes)
    
    # Build embeddings for non-runway similarity search on the node "id" column.
    model_id, id_embeddings = build_id_embeddings(df)
    
    # List to collect meta table rows.
    meta_rows = []
    
    # --- Process transcript files ---
    test_file_paths = glob.glob('/home/yp6443/research/nlp/voice_data/test_file/*.txt')
    file_idx = 1  # adjust as needed

    with open(test_file_paths[file_idx], 'r') as file:
        for line in file:
            line_text = line.strip()
            if not line_text:
                continue
            
            # Extract the time as the first token in the line.
            time_val = line_text.split()[0]
            
            # Run your spaCy NER on the line.
            doc = nlp_ner(line_text)
            print(f"\nLine: \"{line_text}\"")
            print("Entities found:")
            for ent in doc.ents:
                print(f"  - {ent.text} : {ent.label_}")
            
            # Initialize fields (if an entity is missing, the field remains empty).
            callsign = ""
            destination_text = ""
            ac_state_list = []
            
            # Extract entity values (adjust label names as used by your model).
            for ent in doc.ents:
                if ent.label_ == "CALLSIGN":
                    callsign = ent.text
                elif ent.label_ == "DESTINATION":
                    destination_text = ent.text
                elif ent.label_ == "ACSTATE":  # or "AC_STATE" if that is your model's label
                    ac_state_list.append(ent.text)
            ac_state = ",".join(ac_state_list)
            
            # --- Determine the destination runway and final destination ---
            # First, try applying the regex to the entire line.
            designators_line = extract_all_runway_designators(line_text)
            if designators_line:
                # If found in the entire line, choose the last designator.
                chosen = designators_line[-1]
                dest_runway = chosen
                entry_node = find_runway_entry_node(df, chosen)
                if entry_node:
                    final_destination = entry_node
                else:
                    final_destination = f"No entry found for runway {chosen}"
            else:
                # Otherwise, fall back to checking the DESTINATION entity.
                if destination_text and is_runway_pattern(destination_text):
                    designators = extract_all_runway_designators(destination_text)
                    chosen = designators[-1] if designators else None
                    if chosen:
                        dest_runway = chosen
                        entry_node = find_runway_entry_node(df, chosen)
                        if entry_node:
                            final_destination = entry_node
                        else:
                            final_destination = f"No entry found for runway {chosen}"
                    else:
                        final_destination = destination_text
                        dest_runway = ""
                else:
                    dest_runway = ""
                    # For non-runway orders, use the destination_text.
                    final_destination = destination_text
            
            # Build a meta table row.
            meta_row = {
                "callsign": callsign,
                "time": time_val,
                "ac_state": ac_state,
                "dest_runway": dest_runway,
                "destination": final_destination
            }
            meta_rows.append(meta_row)
    
    # Create and print the meta table.
    meta_df = pd.DataFrame(meta_rows, columns=["callsign", "time", "ac_state", "dest_runway", "destination"])
    
    # -----------------------
    # Post-Processing Step:
    # For each callsign, if a dest_runway is provided on any row, propagate it to all rows of that callsign.
    # Replace empty strings with NaN for proper filling.
    meta_df['dest_runway'] = meta_df['dest_runway'].replace('', np.nan)
    # For each callsign, forward-fill then backward-fill the dest_runway.
    meta_df['dest_runway'] = meta_df.groupby('callsign')['dest_runway'].transform(lambda x: x.ffill().bfill())
    # Replace NaN back with an empty string if desired.
    meta_df['dest_runway'] = meta_df['dest_runway'].fillna('')
    # -----------------------
    meta_df = meta_df[~meta_df['time'].isin(['N/A'])]
    
    print("\nFinal Meta Table:")
    print(meta_df.reset_index(drop=True))



Line: "N/A Tokyo Tower, Japan Air 516, spot 18."
Entities found:
  - Japan Air 516 : CALLSIGN

Line: "17:43:02 Japan Air 516, Tokyo Tower, good evening, runway 34R, continue approach, wind 320 at 7, we have departure."
Entities found:
  - Japan Air 516 : CALLSIGN
  - approach : ACSTATE
  - departure : ACSTATE

Line: "17:43:12 Japan Air 516, continue approach, 34R."
Entities found:
  - Japan Air 516 : CALLSIGN
  - approach : ACSTATE
  - 34R. : DESTINATION

Line: "N/A Tokyo Tower, Delta 276 with you on C, proceeding to holding point, 34R."
Entities found:
  - Delta 276 : CALLSIGN
  - proceeding : ACSTATE
  - holding : ACSTATE
  - 34R. : DESTINATION

Line: "17:43:26 Delta 276, Tokyo Tower, good evening, taxi to holding point C1."
Entities found:
  - Delta 276 : CALLSIGN
  - taxi : ACSTATE
  - holding point C1 : DESTINATION

Line: "N/A Holding point C1, Delta 276."
Entities found:
  - Holding point C1 : DESTINATION
  - Delta 276 : CALLSIGN

Line: "17:44:56 Japan Air 516, runway 34R, clear

In [22]:
print("Meta Table:")
meta_df

Meta Table:


,callsign,time,ac_state,dest_runway,destination
1,Japan Air 516,17:43:02,"approach,departure",34R,Rwy_03_001
2,Japan Air 516,17:43:12,approach,34R,Rwy_03_001
4,Delta 276,17:43:26,taxi,34R,holding point C1
6,Japan Air 516,17:44:56,"cleared,land",34R,Rwy_03_001
7,Japan Air 516,17:45:01,"Cleared,land",34R,Rwy_03_001
9,JA722A,17:45:11,taxi,,holding point C5
10,JA722A,17:45:19,Taxi,,holding point C5
12,Japan Air 179,17:45:40,taxi,,holding point C1
15,Japan Air 166,17:45:56,approach,34R,Rwy_03_001
17,Japan Air 166,17:47:23,approach,34R,


In [23]:
accident_time = "17:47:30"
accident_callsign = "Japan Air 516"
new_row_1 = {
        "callsign": accident_callsign,
        "time": accident_time,
        "ac_state": "collision",
        "dest_runway": "",      # Could be left empty if not available.
        "destination": ""
        }

meta_df = pd.concat([meta_df, pd.DataFrame([new_row_1])], ignore_index=True)
    
accident_callsign = "JA722A"
new_row_2 = {
        "callsign": accident_callsign,
        "time": accident_time,
        "ac_state": "collision",
        "dest_runway": "",      # Could be left empty if not available.
        "destination": ""
        }
meta_df = pd.concat([meta_df, pd.DataFrame([new_row_2])], ignore_index=True)

meta_df

,callsign,time,ac_state,dest_runway,destination
0,Japan Air 516,17:43:02,"approach,departure",34R,Rwy_03_001
1,Japan Air 516,17:43:12,approach,34R,Rwy_03_001
2,Delta 276,17:43:26,taxi,34R,holding point C1
3,Japan Air 516,17:44:56,"cleared,land",34R,Rwy_03_001
4,Japan Air 516,17:45:01,"Cleared,land",34R,Rwy_03_001
5,JA722A,17:45:11,taxi,,holding point C5
6,JA722A,17:45:19,Taxi,,holding point C5
7,Japan Air 179,17:45:40,taxi,,holding point C1
8,Japan Air 166,17:45:56,approach,34R,Rwy_03_001
9,Japan Air 166,17:47:23,approach,34R,


In [24]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity


# === Load the airport nodes CSV ===
ICAO = 'HND'
airport_nodes_path = f'./Airport Layouts/{ICAO}_Nodes_Def.csv'
df_nodes = pd.read_csv(airport_nodes_path)
# (Assume df_nodes has a column called "id" that contains node names.)

# === Build embeddings for the airport nodes based on the "id" column ===
# (Assuming your build_id_embeddings function returns a model with an .encode() method and the list/array of embeddings.)
model_id, id_embeddings = build_id_embeddings(df_nodes)
# For example, id_embeddings might be a NumPy array of shape (num_nodes, embedding_dim).

# === Define a function to return the top k most similar node ids given a query string ===
def get_top_k_similar(query, model, embeddings, node_ids, k=3):
    """
    Given a query string, a model, and precomputed embeddings for node_ids,
    returns the top k most similar node ids along with their similarity scores.
    """
    # Compute embedding for the query string:
    query_embedding = model.encode(query)
    
    # Compute cosine similarity between the query and all node embeddings:
    sims = cosine_similarity([query_embedding], embeddings)[0]
    
    # Get indices for the top k similar nodes (sorted by descending similarity):
    top_indices = np.argsort(sims)[::-1][:k]
    
    # Return a list of tuples: (node_id, similarity score)
    return [(node_ids[i], sims[i]) for i in top_indices]

# === Filter the flight log for rows where the destination mentions a holding point ===
# We use case-insensitive matching and ignore rows with missing destination values.
mask = meta_df['destination'].str.contains('holding point', case=False, na=False)
df_holding = meta_df[mask].drop_duplicates(subset=['destination'])

# Get the list of node ids from the airport nodes DataFrame.
node_ids = df_nodes['id'].tolist()

# === For each holding point destination, find the top k similar node names ===
k = 5  # You can adjust this to however many similar names you want to retrieve.
for idx, row in df_holding.iterrows():
    destination_query = row['destination']
    similar_nodes = get_top_k_similar(destination_query, model_id, id_embeddings, node_ids, k=k)
    print(f"Destination '{destination_query}' top {k} similar nodes:")
    for node_name, score in similar_nodes:
        print(f"  {node_name}: {score:.4f}")
    print()


Destination 'holding point C1' top 5 similar nodes:
  Txy_C1_C: 0.4383
  Txy_C2_C: 0.3797
  Rwy_01_02: 0.3454
  Rwy_04_003: 0.3412
  Rwy_02_012: 0.3404

Destination 'holding point C5' top 5 similar nodes:
  Txy_C5_C5B: 0.4252
  Rwy_04_005: 0.3465
  Rwy_02_005: 0.3321
  Rwy_01_005: 0.3223
  Rwy_03_005: 0.3178



In [25]:
from KShortestPaths_TaxiPlanGenerator import *

airports = [
    "AMS", "BOM", "CAI", "CDG", "DEL", "DME", "DXB", "FCO", "FRA", "HKG", "HND",
    "ICN", "IST", "JNB", "KABQ", "KATL", "KBDL", "KBHM", "KBNA", "KBOI", "KBOS",
    "KBTV", "KBUR", "KBWI", "KBZN", "KCHS", "KCLE", "KCLT", "KCRW", "KCVG", "KDAL",
    "KDCA", "KDEN", "KDFW", "KDSM", "KDTW", "KEWR", "KFAR", "KFLL", "KFSD", "KGYY",
    "KHPN", "KIAD", "KIAH", "KICT", "KILG", "KIND", "KISP", "KJAC", "KJAN", "KJAX",
    "KJFK", "KLAS", "KLAX", "KLEX", "KLGA", "KLGB", "KLIT", "KMCO", "KMDW", "KMEM",
    "KMHT", "KMIA", "KMKE", "KMSP", "KMSY", "KOAK", "KOKC", "KOMA", "KONT", "KORD",
    "KPBI", "KPDX", "KPHL", "KPHX", "KPIT", "KPVD", "KPWM", "KSAN", "KSAT", "KSDF",
    "KSEA", "KSFO", "KSJC", "KSLC", "KSNA", "KSTL", "KSWF", "KTEB", "KTPA", "KUL",
    "KVGT", "LHR", "MAD", "MEX", "PANC", "PEK", "PHNL", "PVG", "SIN", "SYD", "TLV",
    "TPE", "YYZ"
]
defFiles = [os.getcwd() + f"/Airport Layouts/{airport}_Nodes_Def.csv" for airport in airports]
linkFiles = [os.getcwd() + f"/Airport Layouts/{airport}_Nodes_Links.csv" for airport in airports]
graphs, nodePositions = loadData(defFiles, linkFiles)

graph = graphs.get(ICAO)
positions = nodePositions.get(ICAO)

# Japan Air 516
start1 = 'Rwy_03_001'
end1 = 'Rwy_03_010'

# JA722A
start2 = 'Txy_H_E'
end2 = 'Txy_C5_C5B'

# generate path from A-star
NUM_PATHS = 3
path1 = aStarMultiple(graph, start1, end1, positions, k=NUM_PATHS)
path2 = aStarMultiple(graph, start2, end2, positions, k=NUM_PATHS)

print('Japan Air 516 Possible Paths:')
for i, (path, distance) in enumerate(path1, start=1):
    print(f"Path {i}: {path}, Distance: {distance:.2f} mile")

print('JA722A Possible Paths:')
for i, (path, distance) in enumerate(path2, start=1):
    print(f"Path {i}: {path}, Distance: {distance:.2f} mile")


Japan Air 516 Possible Paths:
Path 1: ['Rwy_03_001', 'Rwy_03_002', 'Rwy_03_003', 'Rwy_03_004', 'Rwy_03_005', 'Rwy_03_006', 'Rwy_03_007', 'Rwy_03_008', 'Rwy_03_009', 'Rwy_03_010'], Distance: 1.62 mile
Path 2: ['Rwy_03_001', 'Rwy_03_002', 'Rwy_03_003', 'Rwy_03_004', 'Rwy_03_005', 'Rwy_03_006', 'Rwy_03_007', 'Txy_C6_C7', 'Rwy_03_008', 'Rwy_03_009', 'Rwy_03_010'], Distance: 1.62 mile
Path 3: ['Rwy_03_001', 'Rwy_03_002', 'Rwy_03_003', 'Rwy_03_004', 'Rwy_03_005', 'Rwy_03_006', 'Rwy_03_007', 'Rwy_03_008', 'Rwy_03_009', 'Txy_C8_C8B', 'Txy_C8_C', 'Txy_C9_C', 'Txy_C9_C9B', 'Rwy_03_010'], Distance: 2.02 mile
JA722A Possible Paths:
Path 1: ['Txy_H_E', 'Txy_H_C', 'Txy_C5_C5B'], Distance: 0.12 mile
Path 2: ['Txy_H_E', 'Txy_G_E', 'Txy_G_C', 'Txy_C5_C5B'], Distance: 0.21 mile
Path 3: ['Txy_H_E', 'Txy_H_C', 'Txy_G_C', 'Txy_C5_C5B'], Distance: 0.21 mile


In [32]:
import math
import io
import os
import imageio
import matplotlib.pyplot as plt
import pandas as pd

# ----------------------------
# Helper functions
# ----------------------------

def get_position_along_path(path, nodePositions, speed, t):
    """
    Given a list of node IDs in 'path', returns the interpolated position (lat, lon)
    at simulation time t (in seconds), based on the aircraft's speed (mph).
    
    Speed is converted to miles per second (mps) via: mps = speed/3600.
    The function computes cumulative distances along the path and then finds the
    current segment and fraction traveled.
    """
    # Convert mph to miles per second
    mps = speed / 3600.0
    traveled_distance = mps * t

    # Compute cumulative distances along the path segments
    cum_distances = [0]
    for i in range(len(path) - 1):
        pos1 = nodePositions[path[i]]
        pos2 = nodePositions[path[i+1]]
        # Euclidean distance (ensure your coordinates are in a consistent scale)
        segment_distance = math.sqrt((pos2[0] - pos1[0])**2 + (pos2[1] - pos1[1])**2)
        cum_distances.append(cum_distances[-1] + segment_distance)
    total_distance = cum_distances[-1]

    # If we’ve passed the total distance, return the final node position.
    if traveled_distance >= total_distance:
        return nodePositions[path[-1]], total_distance

    # Otherwise, find the segment where the current traveled distance lies.
    for i in range(1, len(cum_distances)):
        if traveled_distance < cum_distances[i]:
            pos1 = nodePositions[path[i-1]]
            pos2 = nodePositions[path[i]]
            seg_distance = cum_distances[i] - cum_distances[i-1]
            seg_progress = (traveled_distance - cum_distances[i-1]) / seg_distance
            lat = pos1[0] + seg_progress * (pos2[0] - pos1[0])
            lon = pos1[1] + seg_progress * (pos2[1] - pos1[1])
            return (lat, lon), total_distance

def draw_background(ax, linksDf, nodePositions, airport_name, xlims=None, ylims=None):
    """
    Draws the airport layout (links and nodes) on the given axis, but only plots those
    nodes and links that are within the provided xlims (longitude limits) and ylims (latitude limits).
    
    Parameters:
      ax           : matplotlib axes on which to draw.
      linksDf      : DataFrame with taxiway/runway links.
      nodePositions: Dictionary of node positions keyed by node ID, with values as (lat, lon).
      airport_name : Name of the airport (used in the title).
      xlims        : Tuple (min_lon, max_lon) to set horizontal boundaries (optional).
      ylims        : Tuple (min_lat, max_lat) to set vertical boundaries (optional).
    """
    
    # Helper function to check if a point is within the limits.
    def is_within(lon, lat):
        within_x = True if xlims is None else (xlims[0] <= lon <= xlims[1])
        within_y = True if ylims is None else (ylims[0] <= lat <= ylims[1])
        return within_x and within_y

    # Draw taxiway/runway links.
    for _, row in linksDf.iterrows():
        # Ensure the coordinates are numbers (in case they were read as strings)
        try:
            n1_lon = float(row['n1.lon'])
            n1_lat = float(row['n1.lat'])
            n2_lon = float(row['n2.lon'])
            n2_lat = float(row['n2.lat'])
        except Exception as e:
            continue  # Skip row if conversion fails

        # Only draw this link if at least one endpoint is within the zoomed area.
        if (xlims is not None or ylims is not None) and not (is_within(n1_lon, n1_lat) or is_within(n2_lon, n2_lat)):
            continue
                
        ax.plot([n1_lon, n2_lon], [n1_lat, n2_lat],
                color='gray', linestyle='-', linewidth=0.5, clip_on=True)

    # Draw nodes and labels only if they are within the specified area.
    for nodeId, (lat, lon) in nodePositions.items():
        try:
            lat = float(lat)
            lon = float(lon)
        except Exception as e:
            continue  # Skip if conversion fails

        if (xlims is not None or ylims is not None) and not is_within(lon, lat):
            continue
        ax.scatter(lon, lat, color='blue', s=20, zorder=2, clip_on=True)
        ax.text(lon, lat, nodeId, fontsize=8, ha='right', va='bottom', clip_on=True)
        
    # Set the axis limits to enforce the zoom.
    if xlims is not None:
        ax.set_xlim(xlims)
    if ylims is not None:
        ax.set_ylim(ylims)
        
    ax.set_xlabel('Longitude [$^\circ$]')
    ax.set_ylabel('Latitude [$^\circ$]')
    ax.set_title(f'{airport_name} Layout')
    ax.grid(False)


def animate_two_aircraft(linksDf, nodePositions, path1, speed1, path2, speed2, airport_name, output_filename, xlims=None, ylims=None):
    """
    Animates two aircraft moving along their respective paths at their given speeds,
    and compiles them into a single animated GIF.
    
    - path1, speed1: Path and speed (mph) for Japan Air 516.
    - path2, speed2: Path and speed (mph) for JA722A.
    - xlims: Optional tuple (min_lon, max_lon) to set the horizontal zoom.
    - ylims: Optional tuple (min_lat, max_lat) to set the vertical zoom.
    """
    
    # Helper to compute total path distance and travel time (in seconds)
    def compute_total_time(path, speed):
        total_distance = 0
        for i in range(len(path) - 1):
            pos1 = nodePositions[path[i]]
            pos2 = nodePositions[path[i+1]]
            segment_distance = math.sqrt((pos2[0] - pos1[0])**2 + (pos2[1] - pos1[1])**2)
            total_distance += segment_distance
        # Total travel time (in seconds) = (total_distance [miles] / speed [mph]) * 3600
        return int(math.ceil((total_distance / speed) * 3600))
    
    total_time1 = compute_total_time(path1, speed1)
    total_time2 = compute_total_time(path2, speed2)
    overall_total_time = max(total_time1, total_time2)

    frames = []
    for t in range(overall_total_time + 1):
        # Compute positions for both aircraft at time t.
        pos1, _ = get_position_along_path(path1, nodePositions, speed1, t)
        pos2, _ = get_position_along_path(path2, nodePositions, speed2, t)
        
        # Create a new figure for this frame.
        fig, ax = plt.subplots(figsize=(10, 10))
        # Set custom zoom (if provided) to focus on the area of interest.
        if xlims is not None:
            ax.set_xlim(xlims)
        if ylims is not None:
            ax.set_ylim(ylims)
            
        draw_background(ax, linksDf, nodePositions, airport_name, xlims=None, ylims=None)
        
        # Plot the planned paths.
        path1_lats = [nodePositions[node][0] for node in path1]
        path1_lons = [nodePositions[node][1] for node in path1]
        ax.plot(path1_lons, path1_lats, color='red', linestyle='--', linewidth=2, label='Japan Air 516')
        
        path2_lats = [nodePositions[node][0] for node in path2]
        path2_lons = [nodePositions[node][1] for node in path2]
        ax.plot(path2_lons, path2_lats, color='green', linestyle='--', linewidth=2, label='JA722A')
        
        # Plot the current positions.
        ax.scatter(pos1[1], pos1[0], color='red', s=50, zorder=3, label='Japan Air 516' if t==0 else "")
        ax.scatter(pos2[1], pos2[0], color='green', s=50, zorder=3, label='JA722A' if t==0 else "")
        
        # Display simulation time.
        ax.text(0.05, 0.95, f"t = {t} s", transform=ax.transAxes,
                fontsize=12, color='black', verticalalignment='top')
        
        # Add legend (only once).
        ax.legend(loc='upper right')
        
        # Save the frame to a buffer.
        buf = io.BytesIO()
        plt.savefig(buf, format='png')
        plt.close(fig)
        buf.seek(0)
        image = imageio.imread(buf)
        frames.append(image)

    # Save all frames as an animated GIF.
    imageio.mimsave(output_filename, frames, duration=0.1)
    print(f"Animation saved to {output_filename}")

# ----------------------------
# Example usage
# ----------------------------

# Define your airport identifier (e.g., ICAO code)
# Load the airport layout data.
# The CSV is assumed to have columns: n1.id, n1.lat, n1.lon, n2.id, n2.lat, n2.lon.
filePath = os.path.join(os.getcwd(), 'Airport Layouts', ICAO + '_Nodes_Links.csv')
linksDf = pd.read_csv(filePath)

# Compute nodePositions from linksDf.
nodes = pd.concat([
    linksDf[['n1.id', 'n1.lat', 'n1.lon']].rename(columns={'n1.id': 'id', 'n1.lat': 'lat', 'n1.lon': 'lon'}),
    linksDf[['n2.id', 'n2.lat', 'n2.lon']].rename(columns={'n2.id': 'id', 'n2.lat': 'lat', 'n2.lon': 'lon'})
]).drop_duplicates()
nodePositions = {row['id']: (row['lat'], row['lon']) for _, row in nodes.iterrows()}

# --- Define Aircraft Paths and Speeds ---

# Japan Air 516 (faster taxi speed)
# (For example, using the first/best path from your previous output.)
path_japan_air = path1[0][0]
speed_japan_air = 5  # mph

# JA722A (slower taxi speed)
path_ja722a = path2[0][0]
speed_ja722a = 1  # mph

# Set the axis limits to zoom in on the area near the holding points.
# Replace these with the appropriate values for your area of interest.
# Example: xlims = (min_lon, max_lon) and ylims = (min_lat, max_lat)
xlims = (139.79, 139.81)  # Example longitude limits
ylims = (35.535, 35.56)    # Example latitude limits
# xlims = None # Example longitude limits
# ylims = None    # Example latitude limits

# Create a single animation showing both aircraft.
animate_two_aircraft(
    linksDf=linksDf,
    nodePositions=nodePositions,
    path1=path_japan_air,
    speed1=speed_japan_air,
    path2=path_ja722a,
    speed2=speed_ja722a,
    airport_name=ICAO,
    output_filename='case-study-1.gif',
    xlims=xlims,
    ylims=ylims
)

/tmp/ipykernel_3166560/2175594405.py:180: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  image = imageio.imread(buf)


Animation saved to case-study-1.gif
